# A Simple Compiler for a Fragment of  `C`

This file shows how a simple compiler for a fragment of the programming language `C` can be implemented using `Larkita`.

First, we describe the grammar of our language using EBNF-notation, since this enables us to give a very concise presentation of the grammar.
Below, the token 'NUMBER' stands for natural numbers and 'ID' represents identifiers.
```
program: function+
       
function: 'int' ID '(' ('int' ID (',' 'int' ID)*)?  ')' '{' decl* stmnt* '}' 
    
decl: 'int' ID ';'  
    
stmnt: '{' stmnt* '}'
     | ID '=' expr ';' 
     | 'if' '(' boolExpr ')' stmnt 
     | 'if' '(' boolExpr ')' stmnt 'else' stmnt 
     | 'while' '(' boolExpr ')' stmnt 
     | 'return' expr ';'
     | expr ';'             
     
boolExpr: boolExpr ('&&' | '||') boolExpr
        | '!' boolExpr             
        | '(' boolExpr ')'
        | expr ('==' | '!=' | '<=' | '>=' | '<' | '>')  expr 
     
expr: expr ('+' | '-' | '*' | '/' | '%') expr 
    | '(' expr ')' 
    | NUMBER
    | ID ('(' (expr (',' expr)*)? ')')?  
```
This grammar is ambiguous.  In the version of this notebook that is based on `Ply` the ambiguity had been resolved
by specifying the appropriate operator precedences.  `Larkita` offers no precedence declarations for its 
<span style="font-variant: small-caps;">LR</span> parser.  Therefore, we will instead *stratify* the grammar:  we
introduce one syntactical variable per precedence level.  This removes the ambiguity and, as a side effect, makes the
precedences and associativities visible in the grammar itself.

## Specification of the Scanner

Unlike `Ply`, which separates the scanner (`ply.lex`) from the parser (`ply.yacc`), `Larkita` reads a *single* grammar
that specifies both the *tokens* and the *grammar rules*:
- The names of *terminals*, i.e. of the tokens, are written in `UPPER_CASE`.  A terminal is defined either by a
  string enclosed in double quotes or by a regular expression enclosed in forward slashes.
- The names of *syntactical variables* are written in `lower_case`.

We store this grammar in small files in the directory `Grammar`, which are displayed and explained one after the
other.  The files of this section describe the scanner and are joined into the string `terminal_definitions`, while
the files of the next section contain the grammar rules and are joined into the string `syntax`.  At the end, both
strings are concatenated.

In [ ]:
#use "../lib/nb.ml";;
load_css ();;
load_lib "larkita.ml"

The token `NUMBER` specifies a natural number.  Note that the regular expression is enclosed in forward slashes.

In [ ]:
cat "Grammar/number.lark"

Next, we define the tokens for those operator symbols that consist of more than one character.  Strictly
speaking, these definitions are not necessary:  In `Larkita`, a string that occurs literally inside a grammar rule,
for example `"=="`, is turned into an *anonymous terminal* automatically.  However, an anonymous terminal that is
built from a symbol like `"=="` receives a generated name of the form `__ANON_1`, and such a name is of little help
when we inspect the token stream or when we read an error message.  If a terminal with the same definition has been
declared explicitly, `Larkita` uses this terminal instead of creating an anonymous one.  Therefore, the declarations
below only serve to give these tokens readable names.

Note that a terminal can also be defined by a string.  In this case no escaping is needed:  in contrast to the
`Ply` version, where the token `t_OR` had to be defined by the regular expression `r'\|\|'`, we can simply write
`"||"`.  Furthermore, `Larkita` sorts string terminals by their length, so `"!="` is preferred over `"!"` and `"<="` is
preferred over `"<"`.

The operators that consist of a single character need no declaration at all:  the list
```
literals = ['+', '-', '*', '/', '%', '(', ')', '{', '}', ';', '=', '<', '>', '!', ',']
```
that was needed for `Ply` has no counterpart in `Larkita`, since `Larkita` names the corresponding anonymous terminals
`PLUS`, `MINUS`, `STAR`, `LPAR`, `RPAR`, and so on.

In [ ]:
cat "Grammar/operators.lark"

Our version of `C` allows both *single-line comments* and *multi-line comments*.
- The regular expression `/\*(.|\n)*?\*/` recognizes multi-line comments.
  Multi-line comments start with the string `/*` and end with the string `*/`.
  Note the use of the *non-greedy* quantor `*?`.  If we have code like
  ```
  /* blah */ a = 1; /* blub */
  ```
  the greedy quantor would recognize the whole line as one comment. 
- The regular expression `//.*` recognizes single-line comments.
  A single line comment starts with the string `//` and extends to the end of the line.

A terminal can be defined as the alternative of several regular expressions.  As the forward slash is the
delimiter of a regular expression in `Larkita`, every occurrence of the character `/` inside a regular expression has
to be escaped as `\/`.

The directive `%ignore` tells the scanner to discard all tokens of the given type, i.e. comments never show up in
the token stream.  This is the `Larkita` equivalent of the function `t_COMMENT` that returned no token.

In [ ]:
cat "Grammar/comment.lark"

The keywords `'int'`, `'if'`, `'else'`, `'while'`, and `'return'` are syntactically identical to identifiers.
In the `Ply` version of this notebook this problem had been solved via the dictionary
```
Keywords = { 'int'   : 'INT', 
             'if'    : 'IF',
             'else'  : 'ELSE', 
             'while' : 'WHILE', 
             'return': 'RETURN'
           }
```
and the function `t_ID` that looked up every identifier in this dictionary.

`Larkita` performs this test automatically:  If a string terminal like `"if"` is also matched by the regular expression
of another terminal, in our case by the regular expression defining `NAME`, then `Larkita` first scans a `NAME` and
afterwards checks whether the resulting string is one of the keywords.  If it is, the type of the token is changed
accordingly.  This mechanism is known as the *unless* mechanism.  Because of this, the string `iffy` is scanned as a
single `NAME` and not as the keyword `if` followed by the identifier `fy`.

Therefore, all we have to specify is the terminal `NAME`.  The keywords themselves are written as strings inside the
grammar rules of the next section.

In [ ]:
cat "Grammar/name.lark"

White space, i.e. *space characters*, *tabulators*, *carriage returns*, and *newlines* are ignored.
In `Ply` we had needed two declarations for this:  the string `t_ignore` for the first three characters and the
function `t_newline`, which additionally had to increment the line number of the scanner.  In `Larkita`, a single
`%ignore` directive suffices, since `Larkita` keeps track of line numbers and column numbers on its own:  every token
`t` provides the fields `t.line`, `t.column`, `t.end_line`, and `t.end_column`.  Hence there is no need for a
function like `find_column` either.

In [ ]:
cat "Grammar/whitespace.lark"

The function `read_grammar` joins the given files of the directory `Grammar`.  The files of this section form the
complete specification of our scanner.

In [ ]:
let read_grammar (names : string list) : string =
  String.concat "\n" (List.map (fun name -> read_file ("Grammar/" ^ name ^ ".lark")) names)

let terminal_definitions = read_grammar [ "number"; "operators"; "comment"; "name"; "whitespace" ]
;;
print_string terminal_definitions;
flush stdout

## Specification of the Parser

Since `Ply` does not support <span style="font-variant: small-caps;">Ebnf</span>, the grammar had to be
rewritten:  every list had to be expressed by a pair of recursive rules like `param_list` and `ne_param_list`.
`Larkita` supports <span style="font-variant: small-caps;">Ebnf</span> directly, so we can use the quantifiers `*`,
`+`, and `?` as well as parentheses for grouping.  Consequently, the grammar given at the beginning of this notebook
can be used nearly unchanged.  There are only two things that we have to take care of:
1. The rules for `expr` and `bool_expr` are ambiguous.  Since the 
   <span style="font-variant: small-caps;">LR</span> parser of `Larkita` supports no precedence declarations,
   we have to encode the operator precedences into the grammar by introducing one syntactical variable per
   precedence level.
2. The grammar determines the abstract syntax tree that the parser returns:  The string literals of an
   alternative form the *head* of the tuple that is built for this alternative, while all other symbols become
   its children.  We choose the literals that are ignored and the *aliases* `-> name`, which replace the head,
   in such a way that the parser returns the nested tuples that are used by our code generator.

A program is a non-empty list of function definitions.  The alias `-> "."` makes the parser return a program as a
tuple of the form `('.', function, ...)`, while the alias `-> "fct"` yields function definitions of the form
`('fct', name, parameters, declarations, statements)`:  The string literals `"int"`, `"("`, `")"`, `"{"`, and `"}"`
would only contribute to the head of the tuple, but the alias replaces this head.

The directive `%ignore: "(", ")", "{", "}", ";"` (note the colon) removes the parentheses, the braces, and the
semicolon from the heads of *all* tuples, since these literals carry no information.

In [ ]:
cat "Grammar/program.lark"

The syntactical variables `parameters`, `declarations`, and `statements` describe lists.  In `Ply` these lists
had to be defined via recursion.  Using <span style="font-variant: small-caps;">Ebnf</span>, one rule per list
suffices.  The children of a repetition are spliced into the tuple of the rule.  Hence the alias `-> "."` yields tuples of
the form `('.', name, ...)` and `('.', statement, ...)`:  Note that the tuples of `parameters` and `declarations` contain
only the tokens of type `NAME`, since all other symbols of these rules are string literals, which only contribute to
the head.

In [ ]:
cat "Grammar/lists.lark"

The rule `statement` has seven alternatives.  As the parentheses, the braces, and the semicolon are ignored,
the heads of the tuples are the keywords `'if'`, `'while'`, and `'return'` and the operator `'='`.  The alternative
for the `if`-`else` statement gets the alias `-> "if-else"`, since otherwise its head would be the tuple
`('if', 'else')`.  A block `"{" statements "}"` and an expression statement `expr ";"` consist of a single
symbol, as their literals are ignored.  Therefore, they return the tuple of this symbol.

The two alternatives `if_stmnt` and `if_else_stmnt` cause the well-known 
[dangling else problem](https://en.wikipedia.org/wiki/Dangling_else):  when the parser has read a statement of the
form `if ( bool_expr ) statement` and the next token is the keyword `else`, then the parser can either *reduce* this
statement or it can *shift* the token `else`.  In the `Ply` version of this notebook this *shift/reduce conflict*
had been resolved by declaring the precedence of the token `ELSE` to be higher than the precedence of the token
`IF` and by attaching the precedence of `IF` to the rule `stmnt : IF '(' bool_expr ')' stmnt` via the keyword
`%prec`.  

`Larkita` resolves a shift/reduce conflict *in favour of shifting*.  This is exactly what we want:  the keyword `else`
is attached to the innermost `if` that does not yet have an `else` part.  Hence a program of the form
```
if (a == 1) if (b == 1) x = 2; else x = 3;
```
is interpreted as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } else {
        x = 3;
    }
}
```
and not as
```
if (a == 1) {
    if (b == 1) {
        x = 2;
    } 
} else {
    x = 3;
}
```

In [ ]:
cat "Grammar/statement.lark"

The `Ply` version of this notebook had used the precedence declarations
```
('left'    , 'OR'),
('left'    , 'AND'),
('right'   , '!'),
('nonassoc', 'EQ', 'NE', 'LE', 'GE', '<', '>'),
```
to disambiguate the rules for `bool_expr`.  We express the same information by *stratifying* the grammar:
- `bool_expr` is a list of `conjunction`s that are separated by the operator `||`,
- a `conjunction` is a list of `negation`s that are separated by the operator `&&`,
- a `negation` is a `bool_atom` that is preceded by an arbitrary number of `!` operators,
- a `bool_atom` is either a comparison of two arithmetical expressions or a `bool_expr` in parentheses.

Since the operators `||` and `&&` are *left associative*, the corresponding rules are *left recursive*:  the
syntactical variable on the left hand side of the rule occurs as the *first* symbol on the right hand side.  The
operator `!` is *right associative*, hence the rule `not_expr` is *right recursive*.  As the comparison operators
are non-associative, the rule `bool_atom` is not recursive at all:  an expression like `a < b < c` is a syntax
error.

The operators are the heads of the tuples, for example the alternative `bool_expr "||" conjunction` yields the
tuple `('||', lhs, rhs)`.  An alternative that consists of a single symbol, like `bool_expr: conjunction`, returns
the tuple of this symbol.  Hence the tuple of the expression `x == 1` is `('==', x, 1)`, where `x` and `1` stand for
the tuples of the variable and of the number, and not a chain of four nested tuples.

The first alternative of `bool_atom` uses an *inner alternative*:  `expr ("==" | "!=" | "<=" | ">=" | "<" | ">") expr`
abbreviates six alternatives, one for every comparison operator.  Hence the comparison operator that has been read
is the head of the tuple.

In [ ]:
cat "Grammar/bool-expr.lark"

The arithmetical expressions are stratified in the same way.  The precedence declarations
```
('left', '+', '-'),
('left', '*', '/', '%')
```
of the `Ply` version are replaced by the three levels `expr`, `product`, and `factor`.  Again, all rules are left
recursive because all arithmetical operators are left associative.  The inner alternatives `("+" | "-")` and
`("*" | "/" | "%")` abbreviate one alternative per operator, so the operator that has been read is the head.

As the parentheses are ignored, the last alternative of the rule `factor` consists of a single symbol.  Therefore, a
parenthesized expression creates no tuple of its own.  A number yields a pair of the form `('NUMBER', digits)` and a
variable yields `('NAME', name)`.  The arguments of a function call are written inline as `[expr ("," expr)*]`:
The children of an optional part and of a repetition are spliced into the tuple of the alternative, so a function
call yields the tuple `('call', ('NAME', name), arg_1, ..., arg_n)`.

In [ ]:
cat "Grammar/expr.lark"

The complete grammar is the concatenation of the grammar rules and the terminal definitions.

In [ ]:
let syntax  = read_grammar [ "program"; "lists"; "statement"; "bool-expr"; "expr" ]
let grammar = syntax ^ "\n" ^ terminal_definitions
;;
print_string grammar;
flush stdout

Now we can create the parser.  `Larkita` always generates a *canonical LR(1)* parser.  The argument
`~start:"program"` declares `program` to be the *start variable* of our grammar.  It could be omitted, since the
start variable defaults to the first rule of the grammar.

`Larkita` writes a description of all states of the LR(1) automaton into the file given by the argument
`~out_file`.  This is the `Larkita` equivalent of the file `parser.out` that had been produced by `Ply`.  If there are
shift/reduce conflicts, `Larkita` prints a warning (in Python, a `ConflictWarning`).  Reduce/reduce conflicts are errors.

In [ ]:
let parser = Larkita.create ~start:"program" ~out_file:"compiler.out" grammar

The only conflict that is reported is the shift/reduce conflict that results from the *dangling else
ambiguity*.  As discussed above, `Larkita` resolves this conflict in favour of shifting and hence in the way that we
want.  All other conflicts have been eliminated by stratifying the grammar.  The following cell prints the section
`Conflicts` of the file `compiler.out`.

In [ ]:
let report = read_file "compiler.out" in
let index sub = let n = String.length sub in let rec find i = if String.sub report i n = sub then i else find (i + 1) in find 0 in
print_string (String.sub report (index "Conflicts\n") (index "States\n" - index "Conflicts\n"));
flush stdout

## Testing the Scanner

The function `Larkita.lex` runs the scanner without invoking the parser.  It returns the list of tokens.
Every token `t` knows its type `t.type_`, its value `t.value`, and its position `t.line` and `t.column`.  Note that
white space and comments do not show up in the resulting list of tokens because of the `%ignore` directives, and
that the keywords have been retyped from `NAME` to `INT`, `IF`, `ELSE`, `WHILE`, and `RETURN`.

In [ ]:
let test_scanner (file_name : string) : token list =
  let program = read_file file_name in
  print_endline program;
  Larkita.lex parser program

In [ ]:
List.iter (fun (t : token) -> Printf.printf "%3d, %3d: %-10s %s\n%!" t.line t.column t.type_ t.value) (test_scanner "Examples/MySum.c")

## Construction of the Abstract Syntax Tree

The parser generated by `Larkita` returns the abstract syntax tree as a nested tuple, i.e. as a value of the type `ast`
(see `../Chapter-07/01-Calculator.ipynb`).  We have chosen the grammar in such a way that these tuples are the nested
tuples that had been computed by the `Ply` version of this notebook:
- a list of *n* items is represented as `('.', item₁, ⋯, itemₙ)`,
- a function definition is represented as `('fct', name, parameters, variables, statements)`,
- a variable is represented as `('NAME', name)` and a number as `('NUMBER', digits)`,
- all other nodes are tuples whose first component is the operator.

The Python version of the code generator works on these nested tuples.  In OCaml, the code generator works on
*abstract syntax trees* that are values of the variant types `expr`, `bool_expr`, and `stmnt` defined below:  The
type of a tree then determines which structures are possible, and the compiler checks that the code generator handles
all of them.  The functions `build_expr`, `build_bool`, `build_stmnt`, `build_function`, and `build_program` convert
the nested tuples into these types.  They take the place of the function `simplify_leaves` of the Python version.

In [ ]:
type arith_op = Add | Sub | Mul | Div | Mod
type cmp_op   = Eq | Ne | Le | Ge | Lt | Gt

type expr =
  | Var    of string                      (* a variable *)
  | Number of string                      (* the digits of a number *)
  | Arith  of arith_op * expr * expr
  | Call   of string * expr list          (* a function call *)

type bool_expr =
  | Compare of cmp_op * expr * expr
  | And     of bool_expr * bool_expr
  | Or      of bool_expr * bool_expr
  | Not     of bool_expr

type stmnt =
  | Block      of stmnt list
  | Assign     of string * expr
  | If         of bool_expr * stmnt
  | If_else    of bool_expr * stmnt * stmnt
  | While      of bool_expr * stmnt
  | Return     of expr
  | Expr_stmnt of expr

type fct = { name : string; parameters : string list; variables : string list; stmnts : stmnt list }

type program = fct list

In [ ]:
let unexpected (t : ast) = invalid_arg ("unexpected tuple in the abstract syntax tree: " ^ Larkita.to_string t)

(* ('NAME', name) -> name *)
let name (t : ast) : string = match t with Tok ("NAME", n) -> n | _ -> unexpected t

let arith_ops = [ ("+", Add); ("-", Sub); ("*", Mul); ("/", Div); ("%", Mod) ]
let cmp_ops   = [ ("==", Eq); ("!=", Ne); ("<=", Le); (">=", Ge); ("<", Lt); (">", Gt) ]

let rec build_expr (t : ast) : expr =
  match t with
  | Node ([ op ], [ lhs; rhs ]) when List.mem_assoc op arith_ops ->
      Arith (List.assoc op arith_ops, build_expr lhs, build_expr rhs)
  | Tok ("NUMBER", digits) -> Number digits
  | Tok ("NAME", n) -> Var n
  | Node ([ "call" ], f :: args) -> Call (name f, List.map build_expr args)
  | _ -> unexpected t

let rec build_bool (t : ast) : bool_expr =
  match t with
  | Node ([ "||" ], [ lhs; rhs ]) -> Or  (build_bool lhs, build_bool rhs)
  | Node ([ "&&" ], [ lhs; rhs ]) -> And (build_bool lhs, build_bool rhs)
  | Node ([ "!" ], [ arg ])       -> Not (build_bool arg)
  | Node ([ op ], [ lhs; rhs ]) when List.mem_assoc op cmp_ops ->
      Compare (List.assoc op cmp_ops, build_expr lhs, build_expr rhs)
  | _ -> unexpected t

let rec build_stmnt (t : ast) : stmnt =
  match t with
  | Node ([ "." ], stmnts) -> Block (List.map build_stmnt stmnts)
  | Node ([ "=" ], [ n; expr ]) -> Assign (name n, build_expr expr)
  | Node ([ "if" ], [ condition; stmnt ]) -> If (build_bool condition, build_stmnt stmnt)
  | Node ([ "if-else" ], [ condition; then_stmnt; else_stmnt ]) ->
      If_else (build_bool condition, build_stmnt then_stmnt, build_stmnt else_stmnt)
  | Node ([ "while" ], [ condition; body ]) -> While (build_bool condition, build_stmnt body)
  | Node ([ "return" ], [ expr ]) -> Return (build_expr expr)
  | expr -> Expr_stmnt (build_expr expr)      (* an expression statement is the tuple of the expression *)

let build_function (t : ast) : fct =
  match t with
  | Node ([ "fct" ], [ n; Node ([ "." ], ps); Node ([ "." ], ds); Node ([ "." ], ss) ]) ->
      { name = name n; parameters = List.map name ps; variables = List.map name ds; stmnts = List.map build_stmnt ss }
  | _ -> unexpected t

let build_program (t : ast) : program =
  match t with
  | Node ([ "." ], functions) -> List.map build_function functions
  | _ -> unexpected t

The function `parse_program` takes a string containing a `C` program, parses it, and returns the abstract
syntax tree of this program as `Some ast`.

If the program contains a syntax error, `Larkita` raises the exception `Larkita.Unexpected_input`.  In `Ply`, the
corresponding error message had been printed by the function `p_error`.  The exception provides the fields
`line` and `column`, while the function `Larkita.get_context` returns the line of the program that contains the error together
with a marker that points at the offending token.  In this case, `parse_program` returns `None`.

In [ ]:
let parse_program (program : string) : program option =
  match Larkita.parse parser program with
  | exception Larkita.Unexpected_input error ->
      Printf.printf "Syntax error in line %d, column %d:\n%!" error.line error.column;
      print_endline (Larkita.get_context error program);
      None
  | ast -> Some (build_program ast)

Below is an example of an erroneous program.

In [ ]:
parse_program "int main() { x = 1 + ; }"

The notebook `AST2Dot.ipynb` provides the function `tuple2dot`.  This function can be used to visualize the
abstract syntax tree that is generated by the function `parse_program`.  The function `program_to_nested` converts
an abstract syntax tree into a nested tuple of the type `nested` from the notebook `AST2Dot.ipynb`.  This is the nested
tuple that the Python version computes.  The function `nested_to_string` from `AST2Dot.ipynb` writes it in the
syntax of Python.

In [ ]:
run_notebook "../AST2Dot.ipynb"

In [ ]:
let rec expr_to_nested (e : expr) : nested =
  let arith_op op = match op with Add -> "+" | Sub -> "-" | Mul -> "*" | Div -> "/" | Mod -> "%" in
  match e with
  | Var x -> Str x
  | Number n -> Tuple ("Number", [ Str n ])
  | Arith (op, lhs, rhs) -> Tuple (arith_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | Call (f, args) -> Tuple ("call", Str f :: List.map expr_to_nested args)

let rec bool_to_nested (b : bool_expr) : nested =
  let cmp_op op = match op with Eq -> "==" | Ne -> "!=" | Le -> "<=" | Ge -> ">=" | Lt -> "<" | Gt -> ">" in
  match b with
  | Compare (op, lhs, rhs) -> Tuple (cmp_op op, [ expr_to_nested lhs; expr_to_nested rhs ])
  | And (lhs, rhs) -> Tuple ("&&", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Or (lhs, rhs) -> Tuple ("||", [ bool_to_nested lhs; bool_to_nested rhs ])
  | Not arg -> Tuple ("!", [ bool_to_nested arg ])

let rec stmnt_to_nested (s : stmnt) : nested =
  match s with
  | Block stmnts -> Tuple (".", List.map stmnt_to_nested stmnts)
  | Assign (x, e) -> Tuple ("=", [ Str x; expr_to_nested e ])
  | If (b, s) -> Tuple ("if", [ bool_to_nested b; stmnt_to_nested s ])
  | If_else (b, s1, s2) -> Tuple ("if-else", [ bool_to_nested b; stmnt_to_nested s1; stmnt_to_nested s2 ])
  | While (b, s) -> Tuple ("while", [ bool_to_nested b; stmnt_to_nested s ])
  | Return e -> Tuple ("return", [ expr_to_nested e ])
  | Expr_stmnt e -> expr_to_nested e

let fct_to_nested (f : fct) : nested =
  let names l = Tuple (".", List.map (fun x -> Str x) l) in
  Tuple ("fct", [ Str f.name; names f.parameters; names f.variables; Tuple (".", List.map stmnt_to_nested f.stmnts) ])

let program_to_nested (p : program) : nested = Tuple (".", List.map fct_to_nested p)

The function `parse` takes a `file_name` as its sole argument.  The file is read and parsed.
The resulting abstract syntax tree is visualized using `graphviz`.  In contrast to the `Ply` version, there is no
need to reset a line counter:  `Larkita` computes the line numbers from the string that it is given, hence the line
numbers that occur in error messages are always correct.

In [ ]:
let parse (file_name : string) : unit =
  let program = read_file file_name in
  print_endline program;
  match parse_program program with
  | None -> print_endline "None"
  | Some ast ->
      print_endline (nested_to_string (program_to_nested ast));
      Graphviz.display (tuple2dot (program_to_nested ast))

In [ ]:
parse "Examples/MySum.c"

## Code Generation

The function `indent` is used to indent the generated assembler commands by preceding them with 8 space characters. 

In [ ]:
let indent (s : string) : string = String.make 8 ' ' ^ s

The function `compile_expr expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents an expression.  
  This abstract syntax tree has the type `expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** The Python function needs a final case `case _: assert False` for nested tuples that are not
expressions.  As the type `expr` contains only expressions, the OCaml compiler checks that every case is handled, and
the final case is not needed.  For the same reason, the Python dictionary `OpToCmd` becomes a `match` on the operator.

In [ ]:
let rec compile_expr (expr : expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Var var -> (* expr is a variable name *)
      let cmd = indent (Printf.sprintf "iload %d" (Dict.find var st)) in
      ([ cmd ], 1)
  | Number n ->
      let cmd = indent ("ldc " ^ n) in
      ([ cmd ], 1)
  | Arith (op, lhs, rhs) ->
      let l1, sz1 = compile_expr lhs st class_name in
      let l2, sz2 = compile_expr rhs st class_name in
      let op_to_cmd = match op with Add -> "iadd" | Sub -> "isub" | Mul -> "imul" | Div -> "idiv" | Mod -> "irem" in
      let cmd = indent op_to_cmd in
      (l1 @ l2 @ [ cmd ], max sz1 (1 + sz2))
  | Call ("println", args) ->
      let cmd_lst   = ref [ indent "getstatic java/lang/System/out Ljava/io/PrintStream;" ] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + 1 + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokevirtual java/io/PrintStream/println(%s)V" (String.make !cnt 'I')) ];
      (!cmd_lst, !stck_size)
  | Call (f, args) ->
      let cmd_lst   = ref [] in
      let stck_size = ref 0 in
      let cnt       = ref 0 in
      List.iter
        (fun arg ->
          let l, sz_arg = compile_expr arg st class_name in
          stck_size := max !stck_size (!cnt + sz_arg);
          cmd_lst   := !cmd_lst @ l;
          cnt       := !cnt + 1)
        args;
      cmd_lst := !cmd_lst @ [ indent (Printf.sprintf "invokestatic %s/%s(%s)I" class_name f (String.make !cnt 'I')) ];
      (!cmd_lst, max !stck_size 1)

The following is a test of the function `compile_expr`.

In [ ]:
let expr = Call ("sum", [ Arith (Add, Var "x", Var "y") ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let expr = Arith (Add, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

The variable `label_counter` is a global counter that is used to create unique label names.
Every call of `new_label` creates a new, unique label.

In [ ]:
let expr = Call ("println", [ Var "x"; Var "y" ])
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_expr expr st "Sum"

In [ ]:
let label_counter = ref 0

let new_label () : string =
  label_counter := !label_counter + 1;
  "l" ^ string_of_int !label_counter

The function `compile_bool expr st class_name` takes three arguments:
- `expr` is an *abstract syntax tree* that represents a Boolean expression.  
  This abstract syntax tree has the type `bool_expr`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

**Type safety:** In Python, Boolean expressions and arithmetical expressions are both nested tuples, so nothing prevents
us from calling `compile_bool` with an arithmetical expression.  In OCaml, they have the different types `bool_expr`
and `expr`, and the type checker rejects such a call.

In [ ]:
let rec compile_bool (expr : bool_expr) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match expr with
  | Compare (op, lhs, rhs) ->
      let op_to_cmd =
        match op with
        | Eq -> "if_icmpeq"
        | Ne -> "if_icmpne"
        | Le -> "if_icmple"
        | Ge -> "if_icmpge"
        | Lt -> "if_icmplt"
        | Gt -> "if_icmpgt"
      in
      let l1, sz1    = compile_expr lhs st class_name in
      let l2, sz2    = compile_expr rhs st class_name in
      let true_label = new_label () in
      let next_label = new_label () in
      let cmd_lst    = l1 @ l2 in
      let cmd        = op_to_cmd in
      let cmd_lst    = cmd_lst @ [ indent (cmd ^ " " ^ true_label) ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 0" ] in
      let cmd_lst    = cmd_lst @ [ indent ("goto " ^ next_label) ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ true_label ^ ":" ] in
      let cmd_lst    = cmd_lst @ [ indent "bipush 1" ] in
      let cmd_lst    = cmd_lst @ [ String.make 4 ' ' ^ next_label ^ ":" ] in
      (cmd_lst, max sz1 (1 + sz2))
  | And (lhs, rhs) | Or (lhs, rhs) ->
      let op_to_cmd = match expr with And _ -> "iand" | _ -> "ior" in
      let l1, sz1   = compile_bool lhs st class_name in
      let l2, sz2   = compile_bool rhs st class_name in
      let cmd       = op_to_cmd in
      let cmd_lst   = l1 @ l2 @ [ indent cmd ] in
      (cmd_lst, max sz1 (1 + sz2))
  | Not arg ->
      let l, sz   = compile_bool arg st class_name in
      let cmd_lst = [ indent "bipush 1" ] @ l @ [ indent "isub" ] in
      (cmd_lst, sz + 1)

Below is a test for the function `compile_bool`.

In [ ]:
let expr = Compare (Eq, Var "x", Var "y")
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Not (Compare (Eq, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

In [ ]:
let expr = Or (Compare (Eq, Var "x", Var "y"), Compare (Lt, Var "x", Var "y"))
let st   = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_bool expr st "Sum"

The function `compile_stmnt stmnt st class_name` takes three arguments:
- `stmnt` is an *abstract syntax tree* that represents a statement.  
  This abstract syntax tree has the type `stmnt`.  
- `st` is short for *symbol table*.  This is a dictionary that maps variable
  names to natural numbers.  Given a variable `x`, the number `Dict.find x st` specifies
  the location where the variable `x` is stored on the stack with respect to the 
  local stack frame.
- `class_name` is the name of the class that is to be generated.

The function returns a pair of the form `(cmds, size)`.
- `cmds` is a list of assembler commands,
- `size` is the maximum size of the stack that is needed. 

The Python version recognizes an expression statement by the fact that no other case matches.  In OCaml, an expression
statement is a value of the form `Expr_stmnt expr`.

In [ ]:
let rec compile_stmnt (stmnt : stmnt) (st : (string, int) Dict.t) (class_name : string) : string list * int =
  match stmnt with
  | Assign (var, expr) ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent (Printf.sprintf "istore %d" (Dict.find var st)) ] in
      (cmd_lst, sz)
  | If (expr, sub_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt sub_stmnt st class_name in
      let else_label = new_label () in
      let lbl_stmnt  = String.make 4 ' ' ^ else_label ^ ":" in
      let cmd_lst    = l1 @ [ indent ("ifeq " ^ else_label) ] @ l2 @ [ lbl_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | If_else (expr, then_stmnt, else_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt then_stmnt st class_name in
      let l3, sz3    = compile_stmnt else_stmnt st class_name in
      let else_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ else_label) in
      let else_stmnt = String.make 4 ' ' ^ else_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ next_label) in
      let cmd_lst    = l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; else_stmnt ] @ l3 @ [ next_stmnt ] in
      (cmd_lst, max sz1 (max sz2 sz3))
  | While (expr, body_stmnt) ->
      let l1, sz1    = compile_bool expr st class_name in
      let l2, sz2    = compile_stmnt body_stmnt st class_name in
      let loop_label = new_label () in
      let next_label = new_label () in
      let if_stmnt   = indent ("ifeq " ^ next_label) in
      let loop_stmnt = String.make 4 ' ' ^ loop_label ^ ":" in
      let next_stmnt = String.make 4 ' ' ^ next_label ^ ":" in
      let goto_stmnt = indent ("goto " ^ loop_label) in
      let cmd_lst    = [ loop_stmnt ] @ l1 @ [ if_stmnt ] @ l2 @ [ goto_stmnt; next_stmnt ] in
      (cmd_lst, max sz1 sz2)
  | Return expr ->
      let cmd_lst, sz = compile_expr expr st class_name in
      let cmd_lst     = cmd_lst @ [ indent "ireturn" ] in
      (cmd_lst, sz)
  | Block stmnt_lst ->
      let cmd_lst = ref [] in
      let size    = ref 0 in
      List.iter
        (fun s ->
          let l, sz = compile_stmnt s st class_name in
          cmd_lst := !cmd_lst @ l;
          size    := max !size sz)
        stmnt_lst;
      (!cmd_lst, !size)
  | Expr_stmnt expr -> (* it is an expression statement *)
      compile_expr expr st class_name

In [ ]:
let stmnt = Assign ("x", Number "42")
let st    = Dict.of_list [ ("x", 0); ("y", 1) ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If (Compare (Eq, Var "x", Var "y"), Assign ("x", Number "42"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = If_else (Compare (Lt, Var "x", Var "y"), Assign ("x", Var "y"), Assign ("y", Var "x"));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = While (Compare (Lt, Var "x", Var "y"), Assign ("x", Arith (Add, Var "x", Number "1")));;
compile_stmnt stmnt st "Sum"

In [ ]:
let stmnt = Block [ Assign ("x", Var "y"); Assign ("x", Number "1"); Assign ("y", Var "x") ];;
compile_stmnt stmnt st "Sum"

In [ ]:
let compile_fct (fct_def : fct) (class_name : string) : string list * int =
  label_counter := 0;
  let { name; parameters; variables; stmnts } = fct_def in
  let m   = List.length parameters in
  let n   = List.length variables in
  let st  = ref Dict.empty in
  let cnt = ref 0 in
  List.iter
    (fun var ->
      st  := Dict.add var !cnt !st;
      cnt := !cnt + 1)
    (parameters @ variables);
  let cmd_lst = ref [] in
  let size    = ref 0 in
  List.iter
    (fun stmnt ->
      let l, sz = compile_stmnt stmnt !st class_name in
      cmd_lst := !cmd_lst @ l;
      size    := max !size sz)
    stmnts;
  let limit_locals = Printf.sprintf ".limit locals %d" (m + n) in
  let limit_stack  = Printf.sprintf ".limit stack  %d" !size in
  let return_stmnt = indent "return" in
  if name <> "main" then begin
    let meth    = Printf.sprintf ".method public static %s(%s)I" name (String.make m 'I') in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ ".end method" ] in
    (cmd_lst, !size)
  end
  else begin
    let meth    = ".method public static main([Ljava/lang/String;)V" in
    let cmd_lst = [ meth; limit_locals; limit_stack ] @ !cmd_lst @ [ return_stmnt; ".end method" ] in
    (cmd_lst, !size)
  end

In [ ]:
let f = { name = "sum"; parameters = [ "x" ]; variables = [ "y"; "z" ]; stmnts = [ Return (Var "x") ] };;
compile_fct f "Sum"

In [ ]:
(* Instead of the Python module os.path, we use the module Filename of the OCaml standard library. *)

In [ ]:
let file = "~/Drive/Kurse/Formal-Languages/Python/Chapter-13/Examples/Primes.c";;
print_endline (Filename.dirname file);;
print_endline (Filename.basename file);;
print_endline (Filename.concat "abc" "xyz.c")

The input to `compile_program(file_name)` is the name of a `C` file.  This file assumed to have the ending `.c`.
This file is compiled and the resulting assembler file is written into a file with the same name but the ending `.jas`.

In [ ]:
let compile_program (file_name : string) : unit =
  let directory = Filename.dirname file_name in
  let base      = Filename.basename file_name in
  let base      = Filename.chop_suffix base ".c" in
  let outfile   = Filename.concat directory (base ^ ".jas") in
  let program   = read_file file_name in
  match parse_program program with
  | None -> ()   (* the syntax error has already been reported *)
  | Some fct_lst ->
      let cmd_lst = ref [] in
      List.iter
        (fun fct ->
          let l, _ = compile_fct fct base in
          cmd_lst := !cmd_lst @ l @ [ "\n" ])
        fct_lst;
      Out_channel.with_open_text outfile (fun handle ->
          output_string handle (".class public " ^ base ^ "\n");
          output_string handle ".super java/lang/Object\n\n";
          output_string handle ".method public <init>()V\n";
          output_string handle "    aload 0\n";
          output_string handle "    invokenonvirtual java/lang/Object/<init>()V\n";
          output_string handle "    return\n";
          output_string handle ".end method\n\n";
          List.iter (fun cmd -> output_string handle (cmd ^ "\n")) !cmd_lst)

The file `Primes.c` is a simple `C` program that computes the prime numbers in a naive way.

In [ ]:
Sys.chdir "Examples";;
Sys.getcwd ()

In [ ]:
shell "ls -l"

In [ ]:
cat "Primes.c"

In [ ]:
compile_program "Primes.c"

In [ ]:
shell "ls -l"

In [ ]:
cat "Primes.jas"

Next, we generate Java byte code using
[jasmin](http://jasmin.sourceforge.net).

In [ ]:
shell "jasmin Primes.jas"

Finally, we run the generated byte code.

In [ ]:
shell "ls -l"

In [ ]:
shell "java Primes"